# ⚖️ Phase 2 — Real-World Legal Benchmark: Stanford LegalBench (CPU-Optimized)
## Evaluating Base vs. Fine-Tuned Models on Actual Federal Court Filings

### 🎯 Research Objective
Previously, we evaluated on synthetic cases and public DBpedia graphs (WebNLG).
Now we test on **actual human-written federal court complaints** from **Stanford's LegalBench** (`ssla_company_defendants` from the Securities Class Action Clearinghouse).

**Core Research Question**:
Does our fine-tuned legal adapter generalize to real-world messy federal complaints (with real docket headers, OCR artifacts, and 100-word run-on sentences) to accurately identify the target corporate defendants?

---
### 💻 CPU-Optimized Execution (Zero GPU/TPU Required)
- **Runtime**: Standard Google Colab **CPU Runtime** (Free & Unlimited).
- **Memory**: `Qwen2.5-1.5B` consumes ~3.1 GB in FP32 (well within Colab's 12.7 GB system RAM).
- **Quantization**: Disabled (pure PyTorch FP32, zero `bitsandbytes` CUDA dependencies).
- **Speed**: ~3 to 5 minutes total for 30 real-world federal complaint cases.

### Step 1: Install Dependencies & CPU Environment Audit

In [ ]:
# Run in Google Colab (CPU mode):
!pip install -q transformers peft accelerate tqdm

In [ ]:
import sys
import os
import json
import re
import time
import psutil
import torch
import transformers
from tqdm import tqdm

print("=" * 65)
print("💻 STANFORD LEGALBENCH EVALUATION — HARDWARE AUDIT")
print("=" * 65)
print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
ram_gb = psutil.virtual_memory().total / (1024**3)
ram_avail = psutil.virtual_memory().available / (1024**3)
print(f"System RAM Total     : {ram_gb:.2f} GB (Available: {ram_avail:.2f} GB)")
if torch.cuda.is_available():
    print(f"Accelerator          : GPU ({torch.cuda.get_device_name(0)})")
else:
    print("Accelerator          : CPU Mode (Optimized FP32 Execution)")
print(f"Transformers Version : {transformers.__version__}")
print("=" * 65)

### Step 2: Locate LegalBench Benchmark Data & Adapter Directory

In [ ]:
# If running in Colab, clone repo if not present
if not os.path.exists("llm-graph-ontology"):
    !git clone https://github.com/sagarsambhwani/Fine-Tuning-PoCs.git
    if os.path.exists("Fine-Tuning-PoCs/llm-graph-ontology"):
        %cd Fine-Tuning-PoCs

def find_path(rel_path):
    candidates = [
        rel_path,
        f"../{rel_path}",
        f"llm-graph-ontology/{rel_path}",
        f"Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/Fine-Tuning-PoCs/llm-graph-ontology/{rel_path}",
        f"/content/llm-graph-ontology/{rel_path}"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Cannot locate path: {rel_path}")

legalbench_file = find_path("data/benchmark/legalbench_real_litigation_30.json")
adapter_dir = find_path("models/qwen-1.5b-legal-graph-adapter")

with open(legalbench_file, "r", encoding="utf-8") as f:
    legalbench_cases = json.load(f)

print(f"✅ Loaded {len(legalbench_cases)} real federal litigation complaints from: {legalbench_file}")
print(f"✅ Located adapter weights at: {adapter_dir}")

### Step 3: Load Base Model on CPU in FP32 (`Qwen2.5-1.5B-Instruct`)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base model ({MODEL_ID}) in CPU FP32 mode...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "cpu"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map=device
)
base_model.eval()
print(f"✅ Base Model Loaded successfully on {device.upper()}!")

### Step 4: Define Inference & Entity Extraction Helper

In [ ]:
def extract_target_defendant(current_model, complaint_text):
    prompt_messages = [
        {
            "role": "system",
            "content": (
                "You are a formal legal entity extractor. "
                "Identify the primary corporate company defendant against whom the legal claims are brought in the complaint excerpt. "
                "State ONLY the exact corporate company name directly. Do not include individual officers, lawyers, or extra explanations."
            )
        },
        {
            "role": "user",
            "content": f"Federal Complaint Excerpt:\n{complaint_text}\n\nWhat is the primary corporate company defendant?"
        }
    ]
    formatted = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(current_model.device)
    with torch.no_grad():
        out = current_model.generate(
            **inputs,
            max_new_tokens=35,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def clean_company_name(name):
    # Remove corporate suffixes and punctuation for clean matching
    cleaned = re.sub(r'\b(inc|corp|corporation|llc|holdings|group|ltd|co)\b', '', name.lower())
    cleaned = re.sub(r'[\W_]+', ' ', cleaned).strip()
    return cleaned

def is_correct_match(pred, target):
    p_clean = clean_company_name(pred)
    t_clean = clean_company_name(target)
    if not p_clean or not t_clean:
        return False
    return (t_clean in p_clean) or (p_clean in t_clean) or (p_clean == t_clean)

print("Inference function and matcher ready.")

### Step 5: Evaluate Untuned Base Model on Stanford LegalBench (30 Real Filings)

In [ ]:
print("=" * 65)
print("🔍 EVALUATING UNTUNED BASE MODEL ON LEGALBENCH (30 REAL COMPLAINTS)")
print("=" * 65)

base_correct = 0
base_results = []
start_time = time.time()

for item in tqdm(legalbench_cases, desc="Base Model CPU"):
    pred = extract_target_defendant(base_model, item['complaint_excerpt'])
    correct = is_correct_match(pred, item['target_company_defendant'])
    if correct:
        base_correct += 1
    base_results.append({
        "id": item['id'],
        "docket": item['docket'],
        "target": item['target_company_defendant'],
        "pred": pred,
        "correct": correct
    })

base_acc = base_correct / len(legalbench_cases)
print(f"\nBase Model Complete in {(time.time() - start_time)/60:.2f} minutes!")
print(f"Base Real-World Party Accuracy: {base_acc * 100:.2f}% ({base_correct}/{len(legalbench_cases)})")

### Step 6: Attach Fine-Tuned Legal Adapter & Evaluate on LegalBench

In [ ]:
print("\n" + "=" * 65)
print("🔍 ATTACHING FINE-TUNED LEGAL ADAPTER (CPU MODE)")
print("=" * 65)

tuned_model = PeftModel.from_pretrained(base_model, adapter_dir)
tuned_model.eval()
print("✅ Fine-tuned legal adapter attached successfully.")

tuned_correct = 0
tuned_results = []
start_time = time.time()

for item in tqdm(legalbench_cases, desc="Tuned Model CPU"):
    pred = extract_target_defendant(tuned_model, item['complaint_excerpt'])
    correct = is_correct_match(pred, item['target_company_defendant'])
    if correct:
        tuned_correct += 1
    tuned_results.append({
        "id": item['id'],
        "docket": item['docket'],
        "target": item['target_company_defendant'],
        "pred": pred,
        "correct": correct
    })

tuned_acc = tuned_correct / len(legalbench_cases)
print(f"\nTuned Model Complete in {(time.time() - start_time)/60:.2f} minutes!")
print(f"Tuned Real-World Party Accuracy: {tuned_acc * 100:.2f}% ({tuned_correct}/{len(legalbench_cases)})")

### Step 7: Print Official Stanford LegalBench Head-to-Head Scorecard

In [ ]:
print("\n" + "=" * 75)
print("📊 OFFICIAL STANFORD LEGALBENCH SCORECARD: REAL FEDERAL LITIGATION")
print("=" * 75)
print(f"{'Model / Setting':<36} | {'Accuracy':<12} | {'Correct / Total':<15}")
print("-" * 75)
print(f"{'1. Untuned Base (Qwen2.5-1.5B Zero-Shot)':<36} | {base_acc*100:>10.2f}% | {f'{base_correct}/{len(legalbench_cases)}':>15}")
print(f"{'2. Fine-Tuned Legal Adapter (Track 1)':<36} | {tuned_acc*100:>10.2f}% | {f'{tuned_correct}/{len(legalbench_cases)}':>15}")
delta = (tuned_acc - base_acc) * 100
print(f"{'Real-World Domain Improvement (Δ)':<36} | {delta:>+10.2f}%")
print("=" * 75)

# Sample qualitative inspection
print("\n🔍 QUALITATIVE SAMPLE INSPECTION:")
for i in range(min(3, len(legalbench_cases))):
    b_item = base_results[i]
    t_item = tuned_results[i]
    print(f"\nCase {i+1} [{b_item['docket']}]:")
    print(f"  Target Answer   : {b_item['target']}")
    print(f"  Base Pred       : {b_item['pred']} [{'CORRECT' if b_item['correct'] else 'WRONG'}]")
    print(f"  Tuned Pred      : {t_item['pred']} [{'CORRECT' if t_item['correct'] else 'WRONG'}]")

export_data = {
    "benchmark": "Stanford_LegalBench_SSLA",
    "total_samples": len(legalbench_cases),
    "hardware": device,
    "base_accuracy": round(base_acc, 4),
    "tuned_accuracy": round(tuned_acc, 4),
    "improvement_delta": round(tuned_acc - base_acc, 4)
}

with open("legalbench_cpu_results.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)
print("\nSaved LegalBench results to legalbench_cpu_results.json")